# Arrow C Data Interface: All Permutations Interop Test

Proves that PyArrow, DataFusion, DuckDB, Polars, and pandas can ALL exchange
Arrow data with each other — zero-copy at a single interop boundary.

**Result: 25/25 permutations pass (5 libraries × 5 libraries).**

In [1]:
import pyarrow as pa
import datafusion
from datafusion import SessionContext
import duckdb
import polars as pl
import pandas as pd

print(f"PyArrow: {pa.__version__}")
print(f"DataFusion: {datafusion.__version__}")
print(f"DuckDB: {duckdb.__version__}")
print(f"Polars: {pl.__version__}")
print(f"pandas: {pd.__version__}")

PyArrow: 20.0.0
DataFusion: 53.0.0
DuckDB: 1.5.4
Polars: 1.34.0
pandas: 2.2.3


## Create reference data in each library

In [2]:
REF_DATA = {"id": [1, 2, 3, 4], "name": ["Alice", "Bob", "Charlie", "Diana"], "score": [95.5, 87.3, 91.2, 88.7]}

# PyArrow
pa_table = pa.table(REF_DATA)
print("PyArrow:", pa_table)

# DataFusion
ctx = SessionContext()
ctx.register_record_batches("src", [pa_table.to_batches()])
df_result = ctx.sql("SELECT * FROM src").to_arrow_table()
print("\nDataFusion:", df_result)

# DuckDB
ddb_result = duckdb.sql("SELECT * FROM pa_table").to_arrow_table()
print("\nDuckDB:", ddb_result)

# Polars
pl_df = pl.DataFrame(REF_DATA)
print("\nPolars:", pl_df)

# pandas
pd_df = pd.DataFrame(REF_DATA)
print("\npandas:")
pd_df

PyArrow: pyarrow.Table
id: int64
name: string
score: double
----
id: [[1,2,3,4]]
name: [["Alice","Bob","Charlie","Diana"]]
score: [[95.5,87.3,91.2,88.7]]

DataFusion: pyarrow.Table
id: int64
name: string
score: double
----
id: [[1,2,3,4]]
name: [["Alice","Bob","Charlie","Diana"]]
score: [[95.5,87.3,91.2,88.7]]

DuckDB: pyarrow.Table
id: int64
name: string
score: double
----
id: [[1,2,3,4]]
name: [["Alice","Bob","Charlie","Diana"]]
score: [[95.5,87.3,91.2,88.7]]

Polars: shape: (4, 3)
┌─────┬─────────┬───────┐
│ id  ┆ name    ┆ score │
│ --- ┆ ---     ┆ ---   │
│ i64 ┆ str     ┆ f64   │
╞═════╪═════════╪═══════╡
│ 1   ┆ Alice   ┆ 95.5  │
│ 2   ┆ Bob     ┆ 87.3  │
│ 3   ┆ Charlie ┆ 91.2  │
│ 4   ┆ Diana   ┆ 88.7  │
└─────┴─────────┴───────┘

pandas:


,id,name,score
0,1,Alice,95.5
1,2,Bob,87.3
2,3,Charlie,91.2
3,4,Diana,88.7


## Cross-library transfers: Every permutation

Each cell shows one library producing Arrow data and another consuming it.

In [3]:
# PyArrow → DataFusion (zero-copy via register_record_batches)
ctx = SessionContext()
ctx.register_record_batches("data", [pa_table.to_batches()])
result = ctx.sql("SELECT * FROM data ORDER BY id").to_arrow_table()
print("PyArrow → DataFusion:", result.column("id").to_pylist())

PyArrow → DataFusion: [1, 2, 3, 4]


In [4]:
# PyArrow → DuckDB (zero-copy via register)
con = duckdb.connect()
con.register("data", pa_table)
result = con.execute("SELECT * FROM data ORDER BY id").to_arrow_table()
print("PyArrow → DuckDB:", result.column("id").to_pylist())

PyArrow → DuckDB: [1, 2, 3, 4]


In [5]:
# PyArrow → Polars (zero-copy via from_arrow)
result = pl.from_arrow(pa_table)
print("PyArrow → Polars:", result["id"].to_list())

PyArrow → Polars: [1, 2, 3, 4]


In [6]:
# DataFusion → DuckDB
ctx = SessionContext()
ctx.register_record_batches("src", [pa_table.to_batches()])
df_out = ctx.sql("SELECT * FROM src WHERE score > 90").to_arrow_table()

con = duckdb.connect()
con.register("df_data", df_out)
result = con.execute("SELECT name FROM df_data").to_arrow_table()
print("DataFusion → DuckDB:", result.column("name").to_pylist())

DataFusion → DuckDB: ['Alice', 'Charlie']


In [7]:
# DuckDB → DataFusion
con = duckdb.connect()
con.register("src", pa_table)
ddb_out = con.execute("SELECT * FROM src WHERE id > 2").to_arrow_table()

ctx = SessionContext()
ctx.register_record_batches("ddb_data", [ddb_out.to_batches()])
result = ctx.sql("SELECT name FROM ddb_data ORDER BY name").to_arrow_table()
print("DuckDB → DataFusion:", result.column("name").to_pylist())

DuckDB → DataFusion: ['Charlie', 'Diana']


In [8]:
# Polars → DataFusion
pl_out = pl_df.filter(pl.col("score") > 89)
pa_from_polars = pl_out.to_arrow()

ctx = SessionContext()
ctx.register_record_batches("pl_data", [pa_from_polars.to_batches()])
result = ctx.sql("SELECT id, name FROM pl_data").to_arrow_table()
print("Polars → DataFusion:", result.column("name").to_pylist())

Polars → DataFusion: ['Alice', 'Charlie']


In [9]:
# DuckDB → Polars
con = duckdb.connect()
con.register("src", pa_table)
ddb_out = con.execute("SELECT * FROM src ORDER BY score DESC").to_arrow_table()
result = pl.from_arrow(ddb_out)
print("DuckDB → Polars:", result["name"].to_list())

DuckDB → Polars: ['Alice', 'Charlie', 'Diana', 'Bob']


In [10]:
# Polars → DuckDB
con = duckdb.connect()
con.register("pl_src", pl_df)
result = con.execute("SELECT name, score FROM pl_src WHERE score > 90").to_arrow_table()
print("Polars → DuckDB:", result.column("name").to_pylist())

Polars → DuckDB: ['Alice', 'Charlie']


In [11]:
# pandas → DataFusion → DuckDB → Polars (chain of 4 libraries)
# This proves the single-layer swap: data flows through 4 engines seamlessly

# 1. pandas creates data
pd_src = pd.DataFrame({"x": [10, 20, 30], "y": ["a", "b", "c"]})

# 2. DataFusion sorts it
ctx = SessionContext()
ctx.register_record_batches("pd_data", [pa.Table.from_pandas(pd_src).to_batches()])
df_sorted = ctx.sql("SELECT * FROM pd_data ORDER BY x DESC").to_arrow_table()

# 3. DuckDB filters it
con = duckdb.connect()
con.register("df_sorted", df_sorted)
ddb_filtered = con.execute("SELECT * FROM df_sorted WHERE x > 10").to_arrow_table()

# 4. Polars consumes it
final = pl.from_arrow(ddb_filtered)
print("pandas → DataFusion → DuckDB → Polars:")
print(final)
print("\n✓ Data flowed through 4 libraries with zero serialization.")

pandas → DataFusion → DuckDB → Polars:
shape: (2, 2)
┌─────┬─────┐
│ x   ┆ y   │
│ --- ┆ --- │
│ i64 ┆ str │
╞═════╪═════╡
│ 30  ┆ c   │
│ 20  ┆ b   │
└─────┴─────┘

✓ Data flowed through 4 libraries with zero serialization.


## Full permutation matrix (automated)

In [12]:
# Run the full 25-permutation test
import subprocess, sys
result = subprocess.run(
    [sys.executable, r"C:\Users\jx815f\Desktop\development\iceberg-notes\pyiceberg_datafusion\arrow_interop_test.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    # Filter out deprecation warnings
    for line in result.stderr.split('\n'):
        if 'DeprecationWarning' not in line and 'fetch_arrow' not in line:
            if line.strip():
                print(line)

Arrow C Data Interface: All Permutations Interop Test

Libraries: PyArrow 20.0.0, DataFusion 53.0.0, DuckDB 1.5.4, Polars 1.34.0, pandas 2.2.3

Testing 5 Ã— 5 = 25 permutations (20 cross-library transfers)

     pyarrow â†’ pyarrow   : âœ“
     pyarrow â†’ datafusion: âœ“
     pyarrow â†’ duckdb    : âœ“
     pyarrow â†’ polars    : âœ“
     pyarrow â†’ pandas    : âœ“

  datafusion â†’ pyarrow   : âœ“
  datafusion â†’ datafusion: âœ“
  datafusion â†’ duckdb    : âœ“
  datafusion â†’ polars    : âœ“
  datafusion â†’ pandas    : âœ“

      duckdb â†’ pyarrow   : âœ“
      duckdb â†’ datafusion: âœ“
      duckdb â†’ duckdb    : âœ“
      duckdb â†’ polars    : âœ“
      duckdb â†’ pandas    : âœ“

      polars â†’ pyarrow   : âœ“
      polars â†’ datafusion: âœ“
      polars â†’ duckdb    : âœ“
      polars â†’ polars    : âœ“
      polars â†’ pandas    : âœ“

      pandas â†’ pyarrow   : âœ“
      pandas â†’ datafusion: âœ“
      pandas â†’ duckdb    : âœ“
      pandas â†’ polars    : â